# Flatmount Mouse Retina RBPMS + FISH Spatial Distribution Analysis

**Goal:** quantify the topographical/spatial distribution of ~50 FISH mRNA probes in flatmount mouse retina images stained with:

- **RBPMS channel** = all retinal ganglion cells (RGCs), used to detect soma/cell positions.
- **FISH probe channel** = one mRNA probe per image, used to estimate per-cell expression.

Each probe should have at least 3 replicate retina images. The notebook outputs:

1. RGC segmentation and per-cell measurements.
2. Per-cell probe expression values and positive/negative calls.
3. Spatial metrics per image and per probe.
4. Simulation/null-model z-scores for whether expression is spatially patterned beyond random placement.
5. Unsupervised ML grouping of probes by spatial pattern.
6. Annotated QC images and CSV/Excel result tables.

**Recommended metric set:** use multiple complementary metrics rather than one metric. A good compact panel is:

- **Moran's I:** spatial autocorrelation; high values mean nearby RGCs have similar expression.
- **Radial slope:** expression bias from central to peripheral retina.
- **Angular harmonic strength:** dorsal/ventral, nasal/temporal, or sector-like angular bias.
- **Spatial entropy:** whether positive cells are broadly distributed or concentrated.
- **Nearest-neighbor ratio:** whether probe-positive cells are clustered relative to all RGCs.
- **Simulation z-scores:** compares the observed metric to randomized positive-cell assignments within the same retina.

## Library roles

- `numpy`: fast numerical arrays and vectorized calculations.
- `pandas`: tables for per-cell, per-image, and per-probe outputs.
- `tifffile`: reads multi-channel TIFF microscopy images.
- `scikit-image`: image preprocessing, threshold segmentation fallback, region measurements, overlays.
- `scipy`: spatial distances, nearest-neighbor calculations, z-scores, and statistics.
- `scikit-learn`: PCA, clustering, scaling, and optional unsupervised ML characterization of probe patterns.
- `matplotlib`: minimal QC plots and final summaries.
- `cellpose` *(optional but recommended)*: ML-based RGC soma segmentation from RBPMS channel.

**Model choice:** Cellpose is used as the default segmentation model because it is a generalist deep-learning cell segmentation model and can produce instance-level soma masks from microscopy images with little training. For best results, manually correct ~10-20 representative RBPMS images and fine-tune Cellpose. If Cellpose is unavailable, this notebook falls back to classical thresholding/watershed-style segmentation, but that is less reliable for dense flatmounts.

## Expected folder structure

Edit this section to match your files. One simple layout is:

```text
project_root/
  raw_images/
    Calb2_rep01.tif
    Calb2_rep02.tif
    Calb2_rep03.tif
    Slc17a6_rep01.tif
    ...
  outputs/
```

Assumptions:

- Each TIFF contains both channels.
- Channel 0 = RBPMS.
- Channel 1 = FISH probe.
- File name before `_rep` is treated as the probe name.

If your microscope saves separate files for each channel, modify `load_two_channel_tiff()`.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import tifffile as tiff

import matplotlib.pyplot as plt

from scipy import ndimage as ndi
from scipy.spatial import cKDTree, distance_matrix
from scipy.stats import zscore, linregress

from skimage import exposure, filters, measure, morphology, segmentation, util
from skimage.color import label2rgb
from skimage.draw import disk
from skimage.feature import peak_local_max

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score

try:
    from cellpose import models
    CELLPOSE_AVAILABLE = True
except Exception:
    CELLPOSE_AVAILABLE = False
    warnings.warn('Cellpose not available. Falling back to classical segmentation.')

PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'raw_images'
OUT_DIR = PROJECT_ROOT / 'outputs'
QC_DIR = OUT_DIR / 'qc_images'
OUT_DIR.mkdir(exist_ok=True, parents=True)
QC_DIR.mkdir(exist_ok=True, parents=True)

RBPMS_CHANNEL = 0
FISH_CHANNEL = 1
PIXEL_SIZE_UM = 1.0      # set your microscope calibration
MIN_CELL_AREA = 25       # pixels; tune based on magnification
MAX_CELL_AREA = 2000     # pixels; tune based on magnification
CELLPOSE_DIAMETER = 14   # pixels; tune to RGC soma diameter
FISH_POSITIVE_METHOD = 'image_otsu'  # 'image_otsu' or 'cell_quantile'
CELL_QUANTILE = 0.90
N_SIM = 200              # increase to 1000 for final analysis
RANDOM_SEED = 7
rng = np.random.default_rng(RANDOM_SEED)

## 1. Build an image manifest

This table identifies each image, probe name, and replicate number. It keeps analysis reproducible and makes batch processing simple.

In [ ]:
def parse_probe_and_rep(path: Path):
    stem = path.stem
    m = re.match(r'(.+?)_rep(\d+)', stem, flags=re.IGNORECASE)
    if m:
        return m.group(1), int(m.group(2))
    return stem, np.nan

image_paths = sorted(list(RAW_DIR.glob('*.tif')) + list(RAW_DIR.glob('*.tiff')))
manifest = pd.DataFrame([
    {'image_path': str(p), 'image_id': p.stem, 'probe': parse_probe_and_rep(p)[0], 'replicate': parse_probe_and_rep(p)[1]}
    for p in image_paths
])

print(f'Found {len(manifest)} images')
display(manifest.head())
manifest.to_csv(OUT_DIR / 'image_manifest.csv', index=False)

## 2. Image loading and channel handling

This function reads TIFFs and returns two 2D arrays: RBPMS and FISH. It handles common TIFF shapes such as `(channels, y, x)`, `(y, x, channels)`, or simple 2-channel stacks.

In [ ]:
def load_two_channel_tiff(path, rbpms_channel=0, fish_channel=1):
    img = tiff.imread(path)
    img = np.asarray(img)

    if img.ndim == 2:
        raise ValueError(f'{path} appears single-channel. Need RBPMS and FISH channels.')

    # Channel-first: C, Y, X
    if img.ndim == 3 and img.shape[0] <= 5:
        rbpms = img[rbpms_channel]
        fish = img[fish_channel]
    # Channel-last: Y, X, C
    elif img.ndim == 3 and img.shape[-1] <= 5:
        rbpms = img[..., rbpms_channel]
        fish = img[..., fish_channel]
    # Z/C/Y/X or T/C/Y/X: use max projection over leading non-channel axis if needed
    elif img.ndim == 4 and img.shape[1] <= 5:
        rbpms = img[:, rbpms_channel].max(axis=0)
        fish = img[:, fish_channel].max(axis=0)
    else:
        raise ValueError(f'Unsupported TIFF shape {img.shape} for {path}')

    rbpms = rbpms.astype(np.float32)
    fish = fish.astype(np.float32)
    return rbpms, fish

# Optional quick check on first image
if len(manifest):
    rbpms0, fish0 = load_two_channel_tiff(manifest.loc[0, 'image_path'])
    print('RBPMS shape:', rbpms0.shape, 'FISH shape:', fish0.shape)

## 3. Segment RBPMS-positive RGC somas

This step detects every RGC soma using the RBPMS channel. Cellpose is preferred for crowded soma segmentation. The fallback method uses background subtraction, thresholding, distance transform, and watershed-like separation.

In [ ]:
if CELLPOSE_AVAILABLE:
    cp_model = models.CellposeModel(gpu=True)  # set gpu=False if needed
else:
    cp_model = None

def normalize_image(img):
    p1, p99 = np.percentile(img, [1, 99.7])
    return exposure.rescale_intensity(img, in_range=(p1, p99), out_range=(0, 1)).astype(np.float32)

def segment_rgc_somas(rbpms_img):
    rb = normalize_image(rbpms_img)

    if CELLPOSE_AVAILABLE:
        masks, flows, styles = cp_model.eval(rb, diameter=CELLPOSE_DIAMETER, channels=[0, 0])
        labels = masks.astype(np.int32)
    else:
        smooth = filters.gaussian(rb, sigma=1.0)
        thresh = filters.threshold_otsu(smooth)
        binary = smooth > thresh
        binary = morphology.remove_small_objects(binary, MIN_CELL_AREA)
        binary = morphology.remove_small_holes(binary, area_threshold=MIN_CELL_AREA)
        dist = ndi.distance_transform_edt(binary)
        coords = peak_local_max(dist, min_distance=max(3, CELLPOSE_DIAMETER // 3), labels=binary)
        markers = np.zeros_like(dist, dtype=np.int32)
        for i, (r, c) in enumerate(coords, start=1):
            markers[r, c] = i
        labels = segmentation.watershed(-dist, markers, mask=binary)

    props = measure.regionprops(labels)
    clean = np.zeros_like(labels, dtype=np.int32)
    new_id = 1
    for p in props:
        if MIN_CELL_AREA <= p.area <= MAX_CELL_AREA:
            clean[labels == p.label] = new_id
            new_id += 1
    return clean

## 4. Measure per-cell FISH expression and retinal coordinates

For each segmented RGC, this computes:

- soma centroid position,
- normalized radial position from retina center,
- polar angle,
- RBPMS intensity,
- FISH intensity,
- binary FISH-positive call.

The normalized coordinates let you compare different retina images despite size differences.

In [ ]:
def estimate_retina_center(labels):
    ys, xs = np.nonzero(labels > 0)
    if len(xs) == 0:
        h, w = labels.shape
        return np.array([w / 2, h / 2]), min(h, w) / 2
    center = np.array([xs.mean(), ys.mean()])
    radius = np.percentile(np.sqrt((xs - center[0])**2 + (ys - center[1])**2), 99)
    return center, radius

def measure_cells(labels, rbpms, fish, image_id, probe, replicate):
    props = measure.regionprops_table(
        labels,
        intensity_image=fish,
        properties=('label', 'area', 'centroid', 'mean_intensity', 'max_intensity')
    )
    df = pd.DataFrame(props).rename(columns={
        'centroid-0': 'y_px',
        'centroid-1': 'x_px',
        'mean_intensity': 'fish_mean',
        'max_intensity': 'fish_max'
    })

    rb_props = measure.regionprops_table(labels, intensity_image=rbpms, properties=('label', 'mean_intensity'))
    rb_df = pd.DataFrame(rb_props).rename(columns={'mean_intensity': 'rbpms_mean'})
    df = df.merge(rb_df, on='label', how='left')

    center, radius = estimate_retina_center(labels)
    dx = df['x_px'].to_numpy() - center[0]
    dy = df['y_px'].to_numpy() - center[1]
    df['r_norm'] = np.sqrt(dx**2 + dy**2) / max(radius, 1)
    df['theta_rad'] = np.arctan2(dy, dx)
    df['x_um'] = df['x_px'] * PIXEL_SIZE_UM
    df['y_um'] = df['y_px'] * PIXEL_SIZE_UM
    df['image_id'] = image_id
    df['probe'] = probe
    df['replicate'] = replicate

    if FISH_POSITIVE_METHOD == 'image_otsu':
        try:
            threshold = filters.threshold_otsu(df['fish_mean'].to_numpy())
        except Exception:
            threshold = np.percentile(df['fish_mean'], 90)
    else:
        threshold = np.quantile(df['fish_mean'], CELL_QUANTILE)

    df['fish_threshold'] = threshold
    df['fish_positive'] = df['fish_mean'] > threshold
    return df

## 5. Spatial metrics and simulation null model

The simulation step asks: *if the same number of cells were called positive, but randomly distributed among RGCs in the same retina, how unusual would the observed spatial pattern be?*

This controls for RGC density and image-specific cell counts.

In [ ]:
def spatial_entropy(x, y, positive, bins=6):
    if positive.sum() < 2:
        return np.nan
    H, _, _ = np.histogram2d(x[positive], y[positive], bins=bins)
    p = H.ravel() / max(H.sum(), 1)
    p = p[p > 0]
    return -np.sum(p * np.log2(p)) / np.log2(bins * bins)

def morans_i(coords, values, k=8):
    n = len(values)
    if n < k + 2 or np.nanstd(values) == 0:
        return np.nan
    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=min(k + 1, n))
    z = values - values.mean()
    num = 0.0
    wsum = 0
    for i in range(n):
        for j in idx[i, 1:]:
            num += z[i] * z[j]
            wsum += 1
    den = np.sum(z**2)
    return (n / wsum) * (num / den) if den > 0 and wsum > 0 else np.nan

def nearest_neighbor_ratio(coords, positive):
    if positive.sum() < 3 or (~positive).sum() < 3:
        return np.nan
    pos_coords = coords[positive]
    all_tree = cKDTree(coords)
    pos_tree = cKDTree(pos_coords)
    pos_nn = pos_tree.query(pos_coords, k=2)[0][:, 1].mean()
    all_nn = all_tree.query(coords, k=2)[0][:, 1].mean()
    return pos_nn / all_nn

def compute_metrics(cell_df):
    coords = cell_df[['x_px', 'y_px']].to_numpy(float)
    expr = cell_df['fish_mean'].to_numpy(float)
    positive = cell_df['fish_positive'].to_numpy(bool)
    r = cell_df['r_norm'].to_numpy(float)
    theta = cell_df['theta_rad'].to_numpy(float)

    expr_z = zscore(expr, nan_policy='omit')
    if np.isnan(expr_z).all():
        expr_z = np.zeros_like(expr)

    radial_slope = linregress(r, expr_z).slope if len(cell_df) > 5 else np.nan
    harmonic1 = np.sqrt(np.mean(expr_z * np.cos(theta))**2 + np.mean(expr_z * np.sin(theta))**2)
    harmonic2 = np.sqrt(np.mean(expr_z * np.cos(2*theta))**2 + np.mean(expr_z * np.sin(2*theta))**2)

    return {
        'n_cells': len(cell_df),
        'n_positive': int(positive.sum()),
        'positive_fraction': float(positive.mean()) if len(positive) else np.nan,
        'morans_i_expr': morans_i(coords, expr_z),
        'radial_slope_expr': radial_slope,
        'angular_harmonic1_expr': harmonic1,
        'angular_harmonic2_expr': harmonic2,
        'positive_entropy': spatial_entropy(cell_df['x_px'].to_numpy(), cell_df['y_px'].to_numpy(), positive),
        'positive_nn_ratio': nearest_neighbor_ratio(coords, positive),
    }

def simulate_metric_zscores(cell_df, n_sim=N_SIM):
    observed = compute_metrics(cell_df)
    metric_names = ['morans_i_expr', 'radial_slope_expr', 'angular_harmonic1_expr', 'angular_harmonic2_expr', 'positive_entropy', 'positive_nn_ratio']
    sim_rows = []
    expr = cell_df['fish_mean'].to_numpy(float).copy()
    positive = cell_df['fish_positive'].to_numpy(bool).copy()

    for _ in range(n_sim):
        temp = cell_df.copy()
        temp['fish_mean'] = rng.permutation(expr)
        temp['fish_positive'] = rng.permutation(positive)
        sim_rows.append(compute_metrics(temp))

    sim = pd.DataFrame(sim_rows)
    out = observed.copy()
    for m in metric_names:
        mu = sim[m].mean(skipna=True)
        sd = sim[m].std(skipna=True)
        out[m + '_null_mean'] = mu
        out[m + '_null_sd'] = sd
        out[m + '_z'] = (observed[m] - mu) / sd if sd and not np.isnan(sd) else np.nan
    return out

## 6. Batch process all images

This is the main loop. It segments RGCs, measures FISH per cell, computes spatial metrics, and saves one annotated QC image per input image.

In [ ]:
def save_qc_image(rbpms, fish, labels, cell_df, out_path):
    base = normalize_image(rbpms)
    overlay = label2rgb(labels, image=base, bg_label=0, alpha=0.25)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(overlay)
    pos = cell_df['fish_positive'].to_numpy(bool)
    ax.scatter(cell_df.loc[~pos, 'x_px'], cell_df.loc[~pos, 'y_px'], s=8, marker='o', label='FISH-')
    ax.scatter(cell_df.loc[pos, 'x_px'], cell_df.loc[pos, 'y_px'], s=10, marker='o', label='FISH+')
    ax.set_title(out_path.stem)
    ax.axis('off')
    ax.legend(loc='lower right', frameon=True)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

all_cells = []
image_metrics = []

for row in manifest.itertuples(index=False):
    print('Processing:', row.image_id)
    rbpms, fish = load_two_channel_tiff(row.image_path)
    labels = segment_rgc_somas(rbpms)
    cells = measure_cells(labels, rbpms, fish, row.image_id, row.probe, row.replicate)
    metrics = simulate_metric_zscores(cells, n_sim=N_SIM)
    metrics.update({'image_id': row.image_id, 'probe': row.probe, 'replicate': row.replicate})

    all_cells.append(cells)
    image_metrics.append(metrics)
    save_qc_image(rbpms, fish, labels, cells, QC_DIR / f'{row.image_id}_qc.png')

cells_df = pd.concat(all_cells, ignore_index=True) if all_cells else pd.DataFrame()
image_metrics_df = pd.DataFrame(image_metrics)

cells_df.to_csv(OUT_DIR / 'per_cell_measurements.csv', index=False)
image_metrics_df.to_csv(OUT_DIR / 'per_image_spatial_metrics.csv', index=False)

display(cells_df.head())
display(image_metrics_df.head())

## 7. Aggregate metrics by probe

This collapses replicate images into one row per probe using the mean and standard deviation of each metric.

In [ ]:
metric_cols = [
    'n_cells', 'n_positive', 'positive_fraction',
    'morans_i_expr', 'radial_slope_expr', 'angular_harmonic1_expr', 'angular_harmonic2_expr',
    'positive_entropy', 'positive_nn_ratio',
    'morans_i_expr_z', 'radial_slope_expr_z', 'angular_harmonic1_expr_z', 'angular_harmonic2_expr_z',
    'positive_entropy_z', 'positive_nn_ratio_z'
]

probe_summary = image_metrics_df.groupby('probe')[metric_cols].agg(['mean', 'std', 'count'])
probe_summary.columns = ['_'.join(c).strip() for c in probe_summary.columns]
probe_summary = probe_summary.reset_index()
probe_summary.to_csv(OUT_DIR / 'per_probe_spatial_metric_summary.csv', index=False)

display(probe_summary.head())

## 8. ML characterization of probe spatial patterns

Because there are about 50 probes and usually no ground-truth spatial class labels, this uses **unsupervised ML**:

- Standardize spatial metrics.
- Reduce to 2 principal components for visualization.
- Cluster probes into candidate spatial-pattern groups.

Interpret clusters biologically after checking the QC images and known retinal axes.

In [ ]:
ml_features = [
    'positive_fraction_mean',
    'morans_i_expr_z_mean',
    'radial_slope_expr_z_mean',
    'angular_harmonic1_expr_z_mean',
    'angular_harmonic2_expr_z_mean',
    'positive_entropy_z_mean',
    'positive_nn_ratio_z_mean'
]

ml_df = probe_summary[['probe'] + ml_features].dropna().copy()
X = ml_df[ml_features].to_numpy()
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
PC = pca.fit_transform(X_scaled)
ml_df['PC1'] = PC[:, 0]
ml_df['PC2'] = PC[:, 1]

# Choose k by silhouette when possible; otherwise default to 3.
candidate_ks = range(2, min(8, len(ml_df)))
scores = {}
for k in candidate_ks:
    labels_k = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init='auto').fit_predict(X_scaled)
    if len(set(labels_k)) > 1:
        scores[k] = silhouette_score(X_scaled, labels_k)

best_k = max(scores, key=scores.get) if scores else min(3, len(ml_df))
ml_df['spatial_cluster'] = KMeans(n_clusters=best_k, random_state=RANDOM_SEED, n_init='auto').fit_predict(X_scaled)

ml_df.to_csv(OUT_DIR / 'probe_spatial_ml_clusters.csv', index=False)
print('Best k:', best_k)
display(ml_df.sort_values('spatial_cluster'))

## 9. Minimal summary plots

These plots summarize the probe-level spatial patterns without excessive image inspection.

In [ ]:
if len(ml_df):
    fig, ax = plt.subplots(figsize=(7, 6))
    for clust, sub in ml_df.groupby('spatial_cluster'):
        ax.scatter(sub['PC1'], sub['PC2'], label=f'cluster {clust}', s=60)
        for _, r in sub.iterrows():
            ax.text(r['PC1'], r['PC2'], r['probe'], fontsize=8)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('Probe spatial-pattern clustering')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'probe_spatial_pca_clusters.png', dpi=200)
    plt.show()

    heat = ml_df.set_index('probe')[ml_features]
    heat_z = pd.DataFrame(StandardScaler().fit_transform(heat), index=heat.index, columns=heat.columns)
    fig, ax = plt.subplots(figsize=(10, max(5, 0.25 * len(heat_z))))
    im = ax.imshow(heat_z, aspect='auto')
    ax.set_yticks(np.arange(len(heat_z.index)))
    ax.set_yticklabels(heat_z.index)
    ax.set_xticks(np.arange(len(heat_z.columns)))
    ax.set_xticklabels(heat_z.columns, rotation=45, ha='right')
    ax.set_title('Standardized spatial metrics by probe')
    fig.colorbar(im, ax=ax, label='z-scored metric')
    fig.tight_layout()
    fig.savefig(OUT_DIR / 'probe_spatial_metric_heatmap.png', dpi=200)
    plt.show()

## 10. Export final Excel workbook

This creates a single workbook with per-cell measurements, per-image metrics, per-probe summaries, and ML clusters.

In [ ]:
excel_path = OUT_DIR / 'flatmount_retina_fish_spatial_analysis.xlsx'
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    manifest.to_excel(writer, sheet_name='manifest', index=False)
    cells_df.to_excel(writer, sheet_name='per_cell_measurements', index=False)
    image_metrics_df.to_excel(writer, sheet_name='per_image_metrics', index=False)
    probe_summary.to_excel(writer, sheet_name='per_probe_summary', index=False)
    ml_df.to_excel(writer, sheet_name='ml_probe_clusters', index=False)

print('Saved:', excel_path)

## How to interpret the main metrics

- **High positive_fraction:** many RGCs express the probe.
- **High Moran's I z-score:** expression is spatially autocorrelated; neighboring RGCs have similar expression more than expected by chance.
- **Positive radial_slope z-score:** expression increases toward peripheral retina. Negative means central enrichment.
- **High angular_harmonic1 z-score:** one-sided angular bias, often compatible with dorsal/ventral or nasal/temporal enrichment depending on orientation.
- **High angular_harmonic2 z-score:** two-lobed or opposing-sector pattern.
- **Low entropy z-score:** positive cells are spatially concentrated.
- **Low nearest-neighbor ratio z-score:** positive cells are more clustered than expected.

For final biological interpretation, orient each retina consistently before analysis. Mark dorsal, ventral, nasal, and temporal directions during imaging or rotate images into a common coordinate frame before running this notebook.